# Rabi Oscillation Experiment Prototype v1
### Outdated version, not compatible with looping

Author: Kyle MacRobbie

### *Setup*

In [251]:
# Imports
import json
import matplotlib.pyplot as plt
import pyvisa
import numpy as np
import scipy
from numpy import random
from __future__ import annotations
from typing import TYPE_CHECKING, Callable
from qcodes.instrument import find_or_create_instrument
from qblox_instruments import Cluster, ClusterType
if TYPE_CHECKING:
    from qblox_instruments.qcodes_drivers.module import Module

In [252]:
# Run to get cluster IP
!qblox-pnp list

Devices:
 - 192.168.137.2: cluster_mm 0.9.1 with name "cluster-mm" and serial number 00015_2251_003


In [253]:
# Connect to cluster
cluster_ip = "192.168.137.2"
cluster_name = "cluster0"
cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=(
        {
            2: ClusterType.CLUSTER_QCM,
            4: ClusterType.CLUSTER_QRM,
            6: ClusterType.CLUSTER_QCM_RF,
        }
        if cluster_ip is None
        else None
    ),
)
cluster.led_brightness('medium') # Sets LED brightness on the modules. Options are 'low', 'medium' and 'high'

# Get modules, and connect to the QRM
def get_connected_modules(cluster: Cluster, filter_fn: Callable | None = None) -> dict[int, Module]:
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True

    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }
modules = get_connected_modules(cluster)
module = list(modules.values())[0]

cluster.led_brightness('medium')

# reset cluster and print cluster status
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, Flags: NONE, Slot flags: NONE


In [254]:
modules
qcm_module = modules[2]
#qrm_module = modules[4]
rf_module = modules[6]

In [255]:
print("\nQCM: {}\nQRM: {}\nRF: {}".format(qcm_module.is_qcm_type, qcm_module.is_qrm_type, qcm_module.is_rf_type))


QCM: True
QRM: False
RF: False


In [256]:
print("\nQCM: {}\nQRM: {}\nRF: {}".format(rf_module.is_qcm_type, rf_module.is_qrm_type, rf_module.is_rf_type))


QCM: True
QRM: False
RF: True


In [257]:
cluster.reset()

In [258]:
rm = pyvisa.ResourceManager()
rm.list_resources()


('ASRL3::INSTR', 'GPIB0::8::INSTR')

In [259]:
scope = rm.open_resource('GPIB0::8::INSTR')
print(scope.query('*IDN?'))

*IDN LECROY,WP715ZI,LCRY0716N47852,8.5.0



### Waveforms

### Sequences

In [260]:
qcm_range = 2.5
awg_offs_range = 32768


offs11 = -0.325
offs12 = -0.395
offs11q1 = round((offs11/qcm_range)*awg_offs_range)
offs12q1 = round((offs12/qcm_range)*awg_offs_range)

ramp13_i = 0/2.5
ramp13_f = 0.07/2.5

offs14q1 = offs11q1

ramp15_i = 0/2.5
ramp15_f = -0.055/2.5

ramp16_i = -0.055/2.5
ramp16_f = 0/2.5


offs21 = -0.25
offs21q1 = round((offs21/qcm_range)*awg_offs_range)

ramp23_i = 0/2.5
ramp23_f = -0.075/2.5

offs24 = -0.325
offs24q1 = round((offs24/qcm_range)*awg_offs_range)

ramp25_i = 0
ramp25_f = 0.055

ramp26_i = 0.055
ramp26_f = 0.075

In [261]:
waveform_length = 5000  # nanoseconds
waveforms_0 = {
	"ramp13" : {
		"data": np.linspace(ramp13_i,ramp13_f,5000).tolist(),
		"index": 13
	},
	"ramp156": {
		"data": np.linspace(ramp15_i,ramp15_f,5000).tolist() + np.linspace(ramp16_i,ramp16_f,5000).tolist(),
		"index": 156
	},
}
waveforms_1 = {
	"ramp23" : {
		"data": np.linspace(ramp23_i,ramp23_f,5000).tolist(),
		"index": 23
	},
	"ramp256": {
		"data": np.linspace(ramp25_i,ramp25_f,5000).tolist() + np.linspace(ramp26_i,ramp26_f,5000).tolist(),
		"index": 256
	},
}
waveforms_rf = {
	"block": {
		"data": [1.0 for i in range(waveform_length)],
		"index": 0
	}
}

In [262]:
# Syncing with other sequencers and resetting the offest to 0 if it is not already 0
seq_qcm0 = f"""
	set_awg_offs	0,0
	upd_param		4
	wait_sync		4
"""

# Step 1: offset to -0.325 V for 5 µs
seq_qcm0 += f"""
	set_awg_offs	{offs11q1},{offs11q1}
	upd_param		5000
"""

# Step 2: offset to -0.395 V for 5 µs
seq_qcm0 += f"""
	set_awg_offs	{offs12q1},{offs12q1}
	upd_param		5000
"""

# # Steps 3-4: ramping and RF pulse
seq_qcm0 += f"""
	play			13,13,5000
	set_awg_offs	{offs14q1},{offs14q1}
	upd_param		5000
"""

# Steps 5-6: two ramps
seq_qcm0 += f"""
	play			156,156,10000
"""

# Resetting offset to 0 and stopping
seq_qcm0 += f"""
	set_awg_offs	0,0
	upd_param		4
	stop
"""

In [263]:
# Syncing with other sequencers and resetting the offest to 0 if it is not already 0
seq_qcm1 = f"""
	set_awg_offs	0,0
	upd_param		4
	wait_sync		4
"""

# Step 1: offset to -0.325 V for 5 µs
seq_qcm1 += f"""
	set_awg_offs	{offs21q1},{offs21q1}
	upd_param		5000
"""

# Step 2: offset to -0.395 V for 5 µs
seq_qcm1 += f"""
	nop
	wait			5000
"""

# Steps 3-4: ramping and RF pulse
seq_qcm1 += f"""
	play			23,23,5000
	set_awg_offs	{offs24q1},{offs24q1}
	upd_param		5000
"""

# Steps 5-6: two ramps
seq_qcm1 += f"""
	play			256,256,10000
"""

# Resetting offset to 0 and stopping
seq_qcm1 += f"""
	set_awg_offs	0,0
	upd_param		4
	stop
"""

In [264]:
seq_rf = f"""
	wait_sync	4
	set_mrk		{0b1111}
	upd_param	15000

	play		0,0,5000

	wait 		10000
	
	set_mrk		{0b0000}
	upd_param	4
	nop
	stop
"""

In [265]:
# Reference 3 GHz so that we can use the frequency mixer and actually see the results on the oscilloscope
seq_reference = f"""
	wait_sync	4
	wait		15000

	play		0,0,5000

	stop
"""

### Upload sequences

In [266]:
sequence_qcm0 = {
    "waveforms": waveforms_0,
    "weights": {},
    "acquisitions": {},
    "program": seq_qcm0,
}
sequence_qcm1 = {
    "waveforms": waveforms_1,
    "weights": {},
    "acquisitions": {},
    "program": seq_qcm1,
}
sequence_rf = {
    "waveforms": waveforms_rf,
    "weights": {},
    "acquisitions": {},
    "program": seq_rf,
}
sequence_reference = {
    "waveforms": waveforms_rf,
    "weights": {},
    "acquisitions": {},
    "program": seq_reference,
}
qcm_module.sequencer0.sequence(sequence_qcm0)
qcm_module.sequencer1.sequence(sequence_qcm1)
rf_module.sequencer0.sequence(sequence_rf)
rf_module.sequencer1.sequence(sequence_reference)

### *Correcting offset, get this data from the characterizing_input_offset tutorial*

### *Function that plays sequence*

In [267]:
def play_sequence(freq_dif):
	qcm_module.disconnect_outputs()
	rf_module.disconnect_outputs()

	qcm_module.sequencer0.connect_out0("I")
	qcm_module.sequencer1.connect_out1("I")
	rf_module.sequencer0.connect_out0(True)
	rf_module.sequencer1.connect_out1(True)

	rf_module.sequencer0.mod_en_awg(True)
	rf_module.out0_lo_en(True)
	rf_module.sequencer0.nco_freq(100e6 + freq_dif)
	rf_module.out0_lo_freq(3e9)

	rf_module.sequencer1.mod_en_awg(True)
	rf_module.out1_lo_en(True)
	rf_module.sequencer1.nco_freq(100e6)
	rf_module.out1_lo_freq(3e9)

	qcm_module.sequencer0.sync_en(True)	# Enable sync
	qcm_module.sequencer1.sync_en(True)	# Enable sync
	rf_module.sequencer0.sync_en(True)	# Enable sync
	rf_module.sequencer1.sync_en(True)	# Enable sync

	qcm_module.arm_sequencer(0) # arm sequencer 0
	qcm_module.arm_sequencer(1) # arm sequencer 1
	rf_module.arm_sequencer(0) # arm sequencer 0
	rf_module.arm_sequencer(1) # arm sequencer 1

	cluster.start_sequencer()
	
	return None

### *Running the experiment*

In [268]:
scope.write(f'TIME_DIV {5e-6} S')
scope.write(f'C1:VOLT_DIV {0.12} V')
scope.write(f'C2:VOLT_DIV {0.12} V')
scope.write(f'TRIG_DELAY -15u')
scope.write(f'TRIG_MODE SINGLE')

18

In [270]:
play_sequence(20e6)

print(qcm_module.get_sequencer_status(0))
print(qcm_module.get_sequencer_status(1))
print(rf_module.get_sequencer_status(0))
print(rf_module.get_sequencer_status(1))


Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: NONE, Warning Flags: NONE, Error Flags: NONE, Log: []


### *Stopping*

In [271]:
qcm_module.stop_sequencer(0)
# qrm_module.stop_sequencer(0)
rf_module.stop_sequencer(0)

# Print status of sequencer.
print(qcm_module.get_sequencer_status(0))
# print(qrm_module.get_sequencer_status(0))
print(rf_module.get_sequencer_status(0))

# Reset the cluster
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, Flags: NONE, Slot flags: NONE
